# Single-Cell Feature Processing

This notebook implements the single-cell feature processing step described in the template paper after Steinbock export.

The exported table `data/cells.csv` contains one row per segmented cell object. Marker intensity columns describe measured IMC signal for each marker, while morphology columns describe object size, shape, and position.

Processing sequence:

1. Filter out cells with area smaller than 4 pixels.
2. Censor each marker intensity column to the 99th percentile.
3. Apply arcsinh transformation to marker columns with cofactor 1.
4. Save the processed single-cell table as a new numbered result.

The raw exported table remains unchanged. The processed table is written separately for downstream analysis.


## Step 0: Configure Workflow Paths

This setup cell defines paths used throughout the notebook. The notebook is designed to run from inside the `notebooks/` directory or from the workflow root.

Important paths:

- `data/cells.csv`: raw Steinbock-exported single-cell table.
- `data/panel.csv`: marker panel used to identify marker intensity columns.
- `scripts/08_process_single_cell_features.py`: reusable processing script.
- `results/11_processed_single_cell_features.csv`: processed output table.


In [1]:
from pathlib import Path
import csv
import subprocess

current = Path.cwd().resolve()
if current.name == 'notebooks':
    WORKFLOW_DIR = current.parent
else:
    WORKFLOW_DIR = current

DATA_DIR = WORKFLOW_DIR / 'data'
RESULTS_DIR = WORKFLOW_DIR / 'results'
SCRIPTS_DIR = WORKFLOW_DIR / 'scripts'

INPUT_CELLS = DATA_DIR / 'cells.csv'
PANEL = DATA_DIR / 'panel.csv'
OUTPUT = RESULTS_DIR / '11_processed_single_cell_features.csv'
SUMMARY = RESULTS_DIR / '11_processed_single_cell_features_summary.csv'

print('Workflow directory:', WORKFLOW_DIR)
print('Input cells table exists:', INPUT_CELLS.exists())
print('Panel exists:', PANEL.exists())
print('Processing script exists:', (SCRIPTS_DIR / '08_process_single_cell_features.py').exists())


Workflow directory: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction
Input cells table exists: True
Panel exists: True
Processing script exists: True


## Step 1: Inspect Exported Single-Cell Inputs

Before feature processing, the exported table should be checked for expected columns.

Expected structure:

- `Image`: source ROI image identifier.
- `Object`: segmented object label within the image.
- Marker columns: one column per marker listed in `data/panel.csv`.
- `area`: segmented object size in pixels.
- Coordinate and shape columns: centroid and morphology measurements from Steinbock region properties.

Marker columns are identified from `data/panel.csv` rather than hard-coded. This keeps the workflow reusable for other IMC datasets with different marker panels.


In [2]:
with INPUT_CELLS.open(newline='', encoding='utf-8') as handle:
    cells_reader = csv.DictReader(handle)
    cells_header = cells_reader.fieldnames or []
    preview_rows = [row for _, row in zip(range(5), cells_reader)]

with PANEL.open(newline='', encoding='utf-8') as handle:
    panel_reader = csv.DictReader(handle)
    marker_columns = [row['name'].strip() for row in panel_reader if row.get('keep', '1') == '1' and row.get('name', '').strip()]

print('Preview rows loaded:', len(preview_rows))
print('Total columns in cells table:', len(cells_header))
print('Marker columns from panel:', len(marker_columns))
print('Area column present:', 'area' in cells_header)
print('First marker columns:', marker_columns[:10])
print('Last table columns:', cells_header[-8:])


Preview rows loaded: 5
Total columns in cells table: 50
Marker columns from panel: 42
Area column present: True
First marker columns: ['ArAr80', 'I1277', 'Xe131', 'Xe134', 'Ba138', 'CD38', 'Perilipin', 'Vimentin', 'B4GALT1', 'MPO']
Last table columns: ['193Ir', 'Pb208', 'area', 'centroid-0', 'centroid-1', 'axis_major_length', 'axis_minor_length', 'eccentricity']


## Step 2: Filter Objects With Area Smaller Than 4 Pixels

The `area` column measures the number of pixels assigned to each segmented object. A valid cell object should occupy a reasonable number of pixels. Objects smaller than 4 pixels are extremely small and are commonly interpreted as segmentation fragments, debris, or technical artifacts.

Template-paper rule:

```text
Remove cells with area < 4 pixels.
```

This filtering step uses morphology information from `regionprops`, not marker intensity values. Rows that fail this criterion are removed before percentile calculation and transformation.


## Step 3: Censor Marker Columns to the 99th Percentile

IMC marker intensity distributions often contain a small number of very high values. These extreme values can dominate downstream scaling, visualization, clustering, or phenotyping. Censoring limits the influence of extreme values while preserving most of the measured distribution.

Censoring procedure:

1. Select one marker column.
2. Calculate the 99th percentile after area filtering.
3. Replace values above that percentile with the percentile value.
4. Repeat independently for every marker column.

Only marker columns are censored. Metadata, object labels, coordinates, area, and shape measurements are not censored.


## Step 4: Apply Arcsinh Transformation With Cofactor 1

After censoring, marker intensity columns are transformed using the inverse hyperbolic sine function. This transformation is commonly used for cytometry and multiplex imaging data because marker intensities are non-negative and often highly skewed.

Formula:

```text
transformed_marker = arcsinh(censored_marker / cofactor)
```

For this template-paper step, the cofactor is 1:

```text
transformed_marker = arcsinh(censored_marker)
```

Effect of the transformation:

- Small values remain close to their original scale.
- Large values are compressed.
- Marker distributions become easier to compare across cells.
- Downstream analysis becomes less dominated by extreme marker values.


## Step 5: Run Single-Cell Feature Processing

The reusable script below performs the full processing sequence in the required order:

```text
filter area < 4
-> censor marker columns at the 99th percentile
-> arcsinh-transform marker columns with cofactor 1
-> save processed single-cell table
```

Output table:

```text
results/11_processed_single_cell_features.csv
```

Processing summary:

```text
results/11_processed_single_cell_features_summary.csv
```


In [3]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '08_process_single_cell_features.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--input-cells', 'data/cells.csv',
    '--panel', 'data/panel.csv',
    '--output', 'results/11_processed_single_cell_features.csv',
    '--summary', 'results/11_processed_single_cell_features_summary.csv',
    '--min-area', '4',
    '--percentile', '99',
    '--cofactor', '1',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)


Input cells: 50877
Retained cells after area filtering: 50849
Removed cells with area < 4: 28
Marker columns processed: 42
Processed table: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/11_processed_single_cell_features.csv
Processing summary: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/11_processed_single_cell_features_summary.csv



## Step 6: Review Processed Outputs

The checks below confirm that the processed table and summary file were created. The summary records how many cells were removed by area filtering, how many marker columns were processed, and the censoring threshold used for each marker.

Expected result:

- The processed table should contain the same non-marker columns as the exported `cells.csv`.
- Marker columns should contain censored and arcsinh-transformed values.
- The number of rows may be lower than the raw export if objects with `area < 4` were present.


In [4]:
with OUTPUT.open(newline='', encoding='utf-8') as handle:
    processed_reader = csv.DictReader(handle)
    processed_header = processed_reader.fieldnames or []
    processed_preview = [row for _, row in zip(range(5), processed_reader)]

with SUMMARY.open(newline='', encoding='utf-8') as handle:
    summary_reader = csv.DictReader(handle)
    summary_rows = list(summary_reader)

print('Processed table exists:', OUTPUT.exists())
print('Processed table path:', OUTPUT)
print('Summary table exists:', SUMMARY.exists())
print('Processed preview rows:', len(processed_preview))
print('Processed columns:', len(processed_header))
print('Summary preview:')
for row in summary_rows[:12]:
    print(row)
print('First processed row preview:')
print(processed_preview[0] if processed_preview else {})


Processed table exists: True
Processed table path: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/11_processed_single_cell_features.csv
Summary table exists: True
Processed preview rows: 5
Processed columns: 50
Summary preview:
{'metric': 'input_cells', 'value': '50877'}
{'metric': 'retained_cells_after_area_filter', 'value': '50849'}
{'metric': 'removed_cells_area_lt_min_area', 'value': '28'}
{'metric': 'min_area_pixels', 'value': '4.0'}
{'metric': 'censoring_percentile', 'value': '99.0'}
{'metric': 'arcsinh_cofactor', 'value': '1.0'}
{'metric': 'marker_columns_processed', 'value': '42'}
{'metric': 'ArAr80_p99_censor_threshold', 'value': '13408.514929501562'}
{'metric': 'I1277_p99_censor_threshold', 'value': '151.09283413478295'}
{'metric': 'Xe131_p99_censor_threshold', 'value': '118.66953366755857'}
{'metric': 'Xe134_p99_censor_threshold', 'value': '54.87642094047423'}
{'metric': 'Ba138_p99_censor_threshold', 'value': '2.986040571212763'}
First processe